<a href="https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

The ranked queue takes the top 50 pages by model_score (from the ML-09
validated model, grouped-split precision@50 = 0.74) and adds two things a
raw score can't give a human reviewer: a reason code (the top 2 features
that pushed this page's score up, in plain language) and an action label
(a single-word-ish category — Fix CTR / snippet, Refresh content, Improve
engagement, Investigate visibility drop, or Review manually — derived
from the single strongest reason).

Observed pattern: in the top 10 rows, "low sessions" and "low scroll
engagement" dominate the reason codes, all mapping to "Improve
engagement." This matches the ML-09 coefficient audit, where users_90d
and sessions_90d were the two largest coefficients in the clean model —
so the reason codes are not inventing a new story, they're surfacing the
same signal the model was already relying on, just per-row instead of
in aggregate.

Caveat: reason codes explain what drove THIS model's score, not a
verified causal reason the page is declining. A page flagged "low
scroll engagement" may simply have a content type where scroll isn't a
meaningful metric (e.g. a short FAQ page) — this is why section 3 (human
review) exists.

In [4]:
%cd /content
!rm -rf flyrank-ml
!git clone https://github.com/uomna/flyrank-ml.git
%cd flyrank-ml
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df_valid = df[df["avg_position"] > 0].copy()

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_valid, groups=df_valid["client_id"]))
train_df = df_valid.iloc[train_idx]

feature_cols_clean = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

X_train_c = train_df[feature_cols_clean]
y_train = train_df["is_declining_label"]

imputer_c = SimpleImputer(strategy="median")
X_train_c_imp = imputer_c.fit_transform(X_train_c)
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c_imp)

log_reg_clean = LogisticRegression(max_iter=1000, random_state=42)
log_reg_clean.fit(X_train_c_scaled, y_train)

# score the FULL valid set (this is the deployment queue, not just the test split)
X_all = df_valid[feature_cols_clean]
X_all_imp = imputer_c.transform(X_all)
X_all_scaled = scaler_c.transform(X_all_imp)
df_valid["model_score"] = log_reg_clean.predict_proba(X_all_scaled)[:, 1]

print("Rows scored:", len(df_valid))
print(df_valid[["content_id", "model_score"]].sort_values("model_score", ascending=False).head())

/content
Cloning into 'flyrank-ml'...
remote: Enumerating objects: 167, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 167 (delta 69), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (167/167), 1.88 MiB | 5.09 MiB/s, done.
Resolving deltas: 100% (69/69), done.
/content/flyrank-ml
Rows scored: 28795
                 content_id  model_score
9678   content_4560b0a818ab     1.000000
16811  content_8e7ba84a972b     0.999956
25590  content_a22b7f6c73c5     0.998062
2797   content_c8ad1f4d0e56     0.998026
25606  content_70b8f5323e29     0.992319


In [5]:
# per-row feature contributions = coefficient * standardized value
contributions = X_all_scaled * log_reg_clean.coef_[0]
contrib_df = pd.DataFrame(contributions, columns=feature_cols_clean, index=df_valid.index)

# map each feature to a human-readable reason phrase
reason_labels = {
    "ctr": "low CTR relative to position",
    "avg_position": "weak search position",
    "users_90d": "low engaged users",
    "sessions_90d": "low sessions",
    "content_age_days": "content is old",
    "days_since_last_update": "not updated recently",
    "days_with_impressions": "inconsistent visibility",
    "days_with_sessions": "inconsistent engagement",
    "scroll_events_90d": "low scroll engagement",
    "engagement_rate": "low engagement rate",
    "scroll_rate": "low scroll rate",
    "pageviews_90d": "low pageviews",
    "clicks_90d": "low clicks",
    "impressions_90d": "low impressions",
    "ai_traffic_pct": "low AI-referral share",
    "ai_sessions_90d": "low AI sessions",
    "engaged_sessions_90d": "low engaged sessions",
    "word_count": "thin content",
    "search_volume": "low search demand",
    "competition": "high keyword competition",
    "cpc": "low commercial value (cpc)"
}

def top_reasons(row_idx, n=2):
    row = contrib_df.loc[row_idx]
    top = row.sort_values(ascending=False).head(n)
    return "; ".join([reason_labels.get(f, f) for f in top.index])

df_valid["reason_code"] = [top_reasons(idx) for idx in df_valid.index]

# simple action label based on the single strongest reason
def action_label(row_idx):
    top_feature = contrib_df.loc[row_idx].sort_values(ascending=False).index[0]
    if top_feature in ["ctr", "avg_position"]:
        return "Fix CTR / snippet"
    elif top_feature in ["content_age_days", "days_since_last_update"]:
        return "Refresh content"
    elif top_feature in ["users_90d", "sessions_90d", "engaged_sessions_90d",
                          "scroll_events_90d", "engagement_rate", "scroll_rate"]:
        return "Improve engagement"
    elif top_feature in ["days_with_impressions", "days_with_sessions"]:
        return "Investigate visibility drop"
    else:
        return "Review manually"

df_valid["action_label"] = [action_label(idx) for idx in df_valid.index]

# the ranked queue: top 50 pages by model_score
ranked_queue = df_valid.sort_values("model_score", ascending=False).head(50)[
    ["content_id", "model_score", "reason_code", "action_label"]
]
print(ranked_queue.head(10).to_string(index=False))

          content_id  model_score                         reason_code       action_label
content_4560b0a818ab     1.000000 low scroll engagement; low sessions Improve engagement
content_8e7ba84a972b     0.999956 low sessions; low scroll engagement Improve engagement
content_a22b7f6c73c5     0.998062 low sessions; low scroll engagement Improve engagement
content_c8ad1f4d0e56     0.998026 low sessions; low scroll engagement Improve engagement
content_70b8f5323e29     0.992319 low sessions; low scroll engagement Improve engagement
content_c53566c0e1b1     0.988907 low sessions; low scroll engagement Improve engagement
content_921b6efc110c     0.983460 low scroll engagement; low sessions Improve engagement
content_3f165aff4181     0.980413 low sessions; low scroll engagement Improve engagement
content_62ed76850efc     0.975180 low sessions; low scroll engagement Improve engagement
content_ea60515fd480     0.971132 low sessions; low scroll engagement Improve engagement


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Intended use:
This ranked queue is a triage aid for a content strategist deciding which
pages to review first — not an auto-publish or auto-refresh pipeline. It
answers "where should a human look first," not "what should change on
the page." The reason codes and action labels are starting hypotheses
for the reviewer to confirm, not instructions to execute blindly.

Who should NOT rely on this directly:
- Anyone auto-triggering content changes off model_score alone
- Anyone using this for a single page's story (the model reasons in
  aggregate, per-row correlations are not per-page causal explanations)
- Anyone outside the training distribution described below

Where it stops being valid:
- Trained and validated on the 30K-row starter CSV (32 clients, trailing
  90-day metrics, one snapshot in time). It has NOT been tested on the
  79M-row warehouse data or on clients outside those 32.
- Precision@50 = 0.74 was measured on ONE grouped train/test split, not
  cross-validated — treat this as a directional estimate, not a fixed
  guarantee.
- No time-based validation exists (the starter CSV has no per-row date),
  so this queue says nothing about how fast a page's status is expected
  to change or how often to re-run.
- Content types with near-100% missing keyword/word-count data (per the
  ML-04 data contract) may get less reliable reason codes, since several
  input features are imputed by median for those rows.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Before acting on any row in this queue, a human must check:

1. Does the reason code make sense for this page's content type?
   (e.g. "low scroll engagement" is meaningless for a short reference
   page — check content_type before assuming the metric applies.)
2. Is the page's data complete, or is the score partly built on imputed
   (median-filled) values? A score built on 5+ imputed features is
   weaker evidence than one built on real measurements.
3. Is this page still live and owned by an active client? The dataset
   is a snapshot — a page may have already been updated, removed, or
   reassigned since the data was pulled.
4. Does the action label match business priority? A page flagged
   "Refresh content" might still be low priority if it has negligible
   impressions_90d — check the raw numbers, not just the label.

The no-go list — never automate these directly from model_score:
- Never auto-publish content changes, auto-delete pages, or auto-adjust
  client-facing metadata (titles, meta descriptions) from this score
  alone — these need a human draft + review step.
- Never use model_score as a client-facing "content quality" grade — it
  is an internal triage signal, not a validated quality metric (same
  caution the FlyRank paper itself gives for its own Health Score).
- Never re-rank or filter using trend_direction or trend_pct even for
  review purposes — that reintroduces the label into the workflow and
  defeats the point of a predictive queue.
- Never treat the top-1 row as a guaranteed win — precision@50 = 0.74
  means roughly 1 in 4 rows in this queue will NOT actually be
  declining; review, don't rubber-stamp.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Since this is not a deployed production system, monitoring here means
lightweight manual checks, not automated alerting infrastructure.

Retrain / re-review triggers:
1. Time-based: re-run the scoring after any new data pull (e.g. once the
   79M-row warehouse becomes the working dataset). A snapshot from
   March 2026 should not silently keep driving decisions in later
   months without a refresh.
2. Precision drift: if a sample of reviewed rows from the queue starts
   showing a hit rate well below the measured 0.74 precision@50 (e.g.
   spot-check 20 acted-on rows monthly and track how many were actually
   declining), that is a signal the model no longer matches current
   patterns and needs re-validation, not just re-scoring.
3. Population shift: if the client mix changes meaningfully (new
   clients added, old ones churned) relative to the 32 clients the
   model was trained on, the grouped-split validation no longer
   describes the current population — retrain and re-validate on the
   new mix.
4. Reason code stability: if reason codes for the top rows suddenly
   shift to a feature family not seen in the ML-09 coefficient audit
   (e.g. cpc or search_volume dominating instead of engagement
   signals), investigate before trusting the new pattern — it could be
   a genuine shift or a data quality issue (e.g. a new silent-null trap
   like the gsc_clicks default-to-0 pattern found in ML-04).

What this section does NOT claim: there is no automated monitoring
pipeline here. This is a documented manual checklist for whoever owns
this queue next, not a running system.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

Exports the top-50 ranked queue (content_id, model_score, reason_code,
action_label) to work/outputs/w07_action_playbook_queue.csv. This file
stays out of git by design (per the CI leak-guard) and regenerates from
this notebook — the paper next week will read this same file rather
than re-deriving the queue from scratch.

In [9]:
import os

os.makedirs("work/outputs", exist_ok=True)

# export the full ranked queue (top 50) with all reason/action columns
ranked_queue.to_csv("work/outputs/w07_action_playbook_queue.csv", index=False)

print("Saved:", "work/outputs/w07_action_playbook_queue.csv")
print("Rows:", len(ranked_queue))
print(ranked_queue.columns.tolist())

Saved: work/outputs/w07_action_playbook_queue.csv
Rows: 50
['content_id', 'model_score', 'reason_code', 'action_label']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.